# 06 · Reliability & Idempotency

> **Module 2 · Reliability & Operations · Lecture 2.4**

In this notebook you will:

- Understand what *idempotent* means, precisely, and why retries are a design input
- Compare `WRITE_TRUNCATE` and `WRITE_APPEND` on a partitioned table
- Build **deterministic row identities** (`partition:order_id`) rather than arrival-time keys
- Demonstrate a real double-count and then prove the invariant that prevents it
- Make backfills safe by scoping writes to the partition being repaired

## 1. The problem retries create

Airflow will retry a failed task. That is a feature — transient failures are
common — but it means **every task must be safe to run more than once**.

Consider the obvious load task:

```python
def load(summary):
    client.insert_rows_json(table, summary_rows)   # appends
```

If the task succeeds but the *worker dies* before the task is marked
successful, Airflow retries it. The rows are inserted twice. The DAG
"succeeded". The numbers are wrong.

This is not a rare race — it is the normal case for a flaky network plus a
long load job.

> **A task is idempotent when running it any number of times leaves the
> destination in the same state as running it once.**

Note the definition says *any number of times*, not *twice*. Retry three
times, backfill a partition, re-run a whole DAG — all of those must be
safe.

## 2. Three tools for idempotency

There are only three strategies that work, and every real design picks one
of them:

| strategy | how | cost | failure mode |
|---|---|---|---|
| **Replace** | truncate the partition, then insert | rewrites data | slower; loses data if you truncate too wide |
| **Merge / upsert** | key-based upsert (`MERGE`, `insert-on-duplicate`) | read-modify-write | more complex; needs a real key |
| **Deduplicate on read** | write everything, collapse by key downstream | storage cost | queries must always dedup |

For BigQuery specifically, the native answer to "replace" is the
**partition decorator**: `WRITE_TRUNCATE` with a partition decorator truncates
*only that partition*, leaving every other partition intact. This is the
single most important property of partitioned BigQuery tables, and it is what
makes daily incremental pipelines safe.

The danger is truncating the *whole table* instead of one partition — which
"works" perfectly on the first run and destroys seven days of data on the
second.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)

from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    partition_for_date,
    dedup_key,
    filter_rows_for_partition,
)
from idempotent_load import apply_load
from dedup_load import dedup_rows

SAMPLE_DIR = COURSE_ROOT / "sample_data"
lines = (SAMPLE_DIR / "orders.csv").read_text(encoding="utf-8").splitlines()

accepted, rejected = transform_orders(lines)
print(f"accepted={len(accepted)} rejected={len(rejected)}")

summary = summarise_by_region(accepted)
print()
print(render_summary_csv(summary))

## 3. Append versus truncate, side by side

`apply_load` in `project/core/idempotent_load.py` implements both write
strategies against a dictionary that stands in for the destination table:

- `mode="append"` — adds the incoming totals to whatever is already there.
- `mode="truncate"` — **replaces** the regions in scope with the incoming
  values; regions not in scope are left alone.

The `partitions` argument is the important one. With `partitions=["AMER"]`,
only `AMER` is replaced. Without it, the scope defaults to every region in
the incoming data, which is still correct but broader.

Now the decisive experiment: run the *same* load twice.

- With `append`, the second run doubles the revenue.
- With `truncate`, the second run produces exactly the same table as the
  first.

That difference is the whole subject of this notebook.

In [ ]:
from idempotent_load import apply_load

# A destination table that already holds yesterday's totals.
existing = {
    "AMER": {"order_count": 300.0, "revenue_eur": 100000.0},
    "APAC": {"order_count": 250.0, "revenue_eur": 90000.0},
}

incoming = {
    "AMER": {"order_count": 318.0, "revenue_eur": 113592.19},
    "APAC": {"order_count": 285.0, "revenue_eur": 108194.92},
}


def show(table, label):
    print(label)
    for region in sorted(table):
        bucket = table[region]
        print(f"  {region:<6} count={bucket['order_count']:>7.1f}  revenue={bucket['revenue_eur']:>12.2f}")
    print()


# --- append ---------------------------------------------------------------
appended = apply_load(existing, incoming, mode="append")
appended_twice = apply_load(appended, incoming, mode="append")
show(appended, "append, once:")
show(appended_twice, "append, twice:")

doubled = appended_twice["AMER"]["revenue_eur"] / appended["AMER"]["revenue_eur"]
print(f"revenue multiplier after one extra run: {doubled:.2f}x")
print("Append is NOT idempotent: a retry silently doubles the revenue.")
print()

# --- truncate -------------------------------------------------------------
truncated = apply_load(existing, incoming, mode="truncate")
truncated_twice = apply_load(truncated, incoming, mode="truncate")
show(truncated, "truncate, once:")
show(truncated_twice, "truncate, twice:")

print("truncate is idempotent:", truncated == truncated_twice)

## 4. Truncate must be scoped

A whole-table truncate is idempotent too — and it is also how you delete
every partition except the one you just loaded.

The safe pattern for a partitioned table is:

```sql
CREATE OR REPLACE TABLE orders PARTITION BY DATE(order_ts) AS ...
-- or, for an append into a partitioned table:
-- WRITE_TRUNCATE with the partition decorator $logical_date
```

The key is that the decorator is derived from the **logical date**, not from
"now". A backfill of `2026-01-15` must truncate `20260115` — not today's
partition — or it will wipe today's freshly loaded data and leave January
stale.

`partition_for_date` derives exactly that decorator, and
`filter_rows_for_partition` selects the rows belonging to it.

In [ ]:
from pipeline_logic import partition_for_date, filter_rows_for_partition
from idempotent_load import apply_load

for logical_date in ("2026-01-15", "2026-01-16", "2026-02-01"):
    print(f"{logical_date} -> partition decorator {partition_for_date(logical_date)}")

print()

# A destination holding three daily partitions.
existing = {
    "AMER": {"order_count": 318.0, "revenue_eur": 113592.19},
    "APAC": {"order_count": 285.0, "revenue_eur": 108194.92},
}

incoming = {
    "AMER": {"order_count": 300.0, "revenue_eur": 101000.00},
    "APAC": {"order_count": 260.0, "revenue_eur": 99000.00},
}

# Scoped replace: only the regions in this partition's scope are touched.
scoped = apply_load(existing, incoming, mode="truncate", partitions=["AMER"])
print("after a scoped replace of AMER only:")
for region in sorted(scoped):
    b = scoped[region]
    print(f"  {region:<6} count={b['order_count']:>7.1f}  revenue={b['revenue_eur']:>12.2f}")
print()
print("APAC survived because it was outside the scope.")
print("That is the whole point of a partition-scoped truncate.")

## 5. Deterministic identity

Truncate/append is one answer. The other is **merge**, and merge needs a
*key*.

The wrong key is a load timestamp:

```python
row["loaded_at"] = datetime.utcnow().isoformat()   # WRONG
```

Every retry produces a *different* key, so deduplication finds no duplicate
and the retry appends a second copy. The deduplication code is correct; the
identity is wrong.

The right key is derived from the data:

```python
def dedup_key(order_id: str, partition: str) -> str:
    return f"{partition}:{order_id}"
```

`20260115:ORD-00042`. It is:

- **deterministic** — the same input always produces the same key;
- **content-derived** — it survives a retry, a re-run, and a backfill;
- **scoped** — the partition is part of the key, so the same `order_id` in a
  different day is a different row (and should be).

Note what is *not* in the key: the run id, the logical date of the *retry*,
the attempt number, the worker hostname. None of those are properties of the
business record.

In [ ]:
from dedup_load import dedup_rows
from pipeline_logic import dedup_key, transform_orders
from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent

SAMPLE_DIR = COURSE_ROOT / "sample_data"
lines = (SAMPLE_DIR / "orders.csv").read_text(encoding="utf-8").splitlines()
accepted, _ = transform_orders(lines)

print("dedup_key('ORD-00042', '20260115') =", dedup_key("ORD-00042", "20260115"))
print()

partition = "20260115"
rows = [
    {"order_id": "ORD-1", "amount_eur": 100.0, "version": 1},
    {"order_id": "ORD-2", "amount_eur": 200.0, "version": 1},
    {"order_id": "ORD-1", "amount_eur": 150.0, "version": 2},
]

result = dedup_rows(rows, partition)
print(f"input rows : {len(rows)}")
print(f"output rows: {len(result)}")
for row in result:
    print(f"  {row}")
print()

print("The duplicate ORD-1 collapsed to its LAST version.")
print("Keys are stable, so running dedup_rows again changes nothing:")
print("  idempotent:", dedup_rows(result, partition) == result)

## 6. The double-count, demonstrated

Now let's put the whole thing together and reproduce the failure that
idempotency exists to prevent.

The scenario:

1. The pipeline loads a partition and **succeeds**.
2. The worker is killed before Airflow can record the success.
3. Airflow retries the task.
4. The append mode loads the same rows again.
5. The DAG reports success; the revenue is doubled.

The demo below runs the load twice and checks an **invariant**: for each
region, the revenue in the destination must equal the revenue in the source
summary. The append path violates it; the truncate path satisfies it.

This invariant check is the real lesson. You do not prevent double-counting
by being careful — you prevent it by **asserting**, on every run, that the
destination matches the source, and failing the task when it does not.

In [ ]:
from idempotent_load import apply_load

SUMMARY = {
    "AMER": {"order_count": 318.0, "revenue_eur": 113592.19},
    "APAC": {"order_count": 285.0, "revenue_eur": 108194.92},
    "EMEA": {"order_count": 322.0, "revenue_eur": 118423.22},
    "LATAM": {"order_count": 320.0, "revenue_eur": 110646.18},
}


def validate_load(destination, source, *, tolerance=0.01):
    """Check that the destination matches the source exactly.

    Args:
        destination: What is actually in the table.
        source: What the pipeline computed.
        tolerance: Allowed absolute difference per region.

    Returns:
        A list of human-readable violations; empty means valid.
    """
    violations = []
    for region, expected in source.items():
        actual = destination.get(region)
        if actual is None:
            violations.append(f"{region}: missing from destination")
            continue
        if abs(actual["revenue_eur"] - expected["revenue_eur"]) > tolerance:
            violations.append(
                f"{region}: revenue {actual['revenue_eur']:.2f} != {expected['revenue_eur']:.2f}"
            )
        if abs(actual["order_count"] - expected["order_count"]) > tolerance:
            violations.append(
                f"{region}: count {actual['order_count']:.1f} != {expected['order_count']:.1f}"
            )
    return violations


# --- the failure mode: append --------------------------------------------
destination = apply_load({}, SUMMARY, mode="append")
destination = apply_load(destination, SUMMARY, mode="append")   # the retry

violations = validate_load(destination, SUMMARY)
print("APPEND path after a retry:")
print(f"  AMER revenue in table: {destination['AMER']['revenue_eur']:.2f}")
print(f"  AMER revenue expected: {SUMMARY['AMER']['revenue_eur']:.2f}")
print(f"  invariant violations : {len(violations)}")
for violation in violations:
    print(f"    - {violation}")
print()
print("The DAG would have reported SUCCESS with wrong numbers.")
print()

# --- the fix: truncate ---------------------------------------------------
destination = apply_load({}, SUMMARY, mode="truncate")
destination = apply_load(destination, SUMMARY, mode="truncate")  # the retry

violations = validate_load(destination, SUMMARY)
print("TRUNCATE path after a retry:")
print(f"  AMER revenue in table: {destination['AMER']['revenue_eur']:.2f}")
print(f"  invariant violations : {len(violations)}")
print()
print("Identical state after one run or two. That is idempotency.")

## 7. Retry policy as a design decision

Retries are not a number you copy from a blog post. They are a decision
about **what kind of failure** you expect.

| failure kind | example | retry? | why |
|---|---|---|---|
| Transient infrastructure | 503 from the API, socket timeout | **yes** | the next attempt usually works |
| Transient dependency | warehouse lock, rate limit | yes, with backoff | needs a delay, not just another try |
| Permanent data error | unparseable date in a row | **no** | retrying produces the same error |
| Deterministic config error | missing Variable, bad path | no | retrying wastes 3 × timeout |

Two rules make retries safe:

1. **Only retry the idempotent part.** Extract and validate before the write;
   if the write fails, retry only the write. A "retry the whole task" policy
   on a task that both reads and writes is fine only if the write is
   idempotent.
2. **Back off, and cap it.** `retry_delay` with exponential backoff and a
   maximum attempt count. Three immediate retries against a rate-limited API
   are three immediate failures.

Also remember `max_active_runs`. A DAG that catches up over thirty days
will launch thirty runs, each retrying — the load on the warehouse scales
with the product of both. Throttling one without the other does not help.

## 8. Backfill safety

A backfill re-runs historical logical dates. It is safe when:

- writes are scoped to the partition derived from the **logical date**;
- row identities are content-derived;
- the DAG does not depend on "now" (no `datetime.now()` in the business
  logic, no reading today's file).

The last one is the subtle one. If a task does
`(COURSE_ROOT / "sample_data" / f"orders_{today}.csv")`, then a backfill of
January reads February's file and writes January's partition — and the
invariant check will catch it, but only if you wrote one.

In [ ]:
from datetime import datetime, timedelta, timezone

from pipeline_logic import partition_for_date

# A simple exponential-backoff retry simulator.
def simulate_retries(attempts_allowed, failures, delay_base=timedelta(minutes=5), max_delay=timedelta(hours=1)):
    """Simulate a retry policy with exponential backoff.

    Args:
        attempts_allowed: Total attempts (1 means no retries).
        failures: How many attempts fail before one succeeds.
        delay_base: The first retry delay.
        max_delay: The cap on the retry delay.

    Returns:
        A dict describing the outcome.
    """
    attempt = 0
    waits = []
    while attempt < attempts_allowed:
        attempt += 1
        if attempt > failures:
            break
        delay = min(delay_base * (2 ** (attempt - 1)), max_delay)
        waits.append(delay)
    return {
        "attempts_used": attempt,
        "succeeded": attempt > failures,
        "waits": waits,
        "total_wait": sum(waits, timedelta()),
    }


for failures in (0, 1, 2, 3):
    result = simulate_retries(attempts_allowed=3, failures=failures)
    print(
        f"{failures} transient failure(s): attempts={result['attempts_used']} "
        f"succeeded={result['succeeded']} total_wait={result['total_wait']}"
    )

print()
print("With retries=2 (3 attempts) and backoff from 5 minutes:")
print("  worst-case wait before the final attempt:", simulate_retries(3, 2)["total_wait"])
print("  a permanent failure costs the full budget every time")
print("  -> validate before you write, so permanent errors fail fast.")

## 9. The reliability checklist

For every task in every DAG:

- [ ] **Is it idempotent?** Run it twice; the destination must be unchanged.
- [ ] **Are writes scoped?** Partition decorator from the logical date, not from now.
- [ ] **Do rows have deterministic identity?** Content-derived, never arrival-derived.
- [ ] **Is there an invariant check after the write?** Fail the task if the destination does not match the source.
- [ ] **Is the retry policy deliberate?** Only transient failures are retried; backoff is exponential and capped.
- [ ] **Does the task read anything time-dependent?** No `now()` in business logic.
- [ ] **Does it pass `execution_timeout`?** A hung task must not hold a worker slot forever.
- [ ] **Is it observable?** The task's logs state the row counts and the partition written, so a human can tell success from silent wrongness.

## Exercises

1. **The invariant.** Write `check_no_double_count(destination, source)` that returns `True` only when every region's revenue in `destination` equals the source's within one cent. Then use `apply_load` to show it passing for `mode="truncate"` after two runs and failing for `mode="append"` after two runs.
2. **Harder — deterministic keys.** Construct three rows for the same `order_id` but different partitions (`20260115`, `20260116`) and show that `dedup_key` produces two distinct keys while three rows in the *same* partition produce one. Explain why the partition must be part of the key.
3. **Harder — backfill safety.** A backfill re-runs logical date `2026-01-10` today. Name three things in a typical DAG that would break under backfill, and say for each what the fix is.

<details>
<summary><b>Solution (click to reveal)</b></summary>

**1.**

```python
def check_no_double_count(destination, source, tolerance=0.01):
    for region, expected in source.items():
        actual = destination.get(region)
        if actual is None:
            return False
        if abs(actual["revenue_eur"] - expected["revenue_eur"]) > tolerance:
            return False
    return True
```

After two `truncate` runs the table equals the source and the check returns `True`; after two `append` runs the revenue is doubled and the check returns `False`.

**2.** `dedup_key("ORD-1", "20260115")` is `"20260115:ORD-1"` and `dedup_key("ORD-1", "20260116")` is `"20260116:ORD-1"` — two distinct keys, so `dedup_rows` keeps both rows. Three rows in partition `20260115` collapse to one. The partition belongs in the key because an order id is only unique *within a day*; the same id appearing on two days is a different business fact, and keying on the id alone would silently discard one day's revenue.

**3.** Three typical breakages:

- *A task reading "today's" file* (`orders_{today}.csv`) reads the wrong day's data and writes it into the backfilled partition. Fix: derive the source path from the logical date or the data interval.
- *A whole-table truncate instead of a partition-scoped one* wipes the freshly loaded current-day partition. Fix: use the partition decorator `partition_for_date(logical_date)` with `WRITE_TRUNCATE`.
- *Row identity based on load time* means the backfilled rows get fresh keys and are appended alongside the originals, doubling the partition. Fix: content-derived `dedup_key(order_id, partition)` plus a merge/upsert or a partition-scoped truncate.

</details>